# Milestone 2: Data Understanding, Quality Assessment & Cleaning
## Electric Vehicle Energy Consumption Prediction

**Project Objectives:**
1. **Data Understanding**: Inspect features, data types, and initial distributions.
2. **Data Quality Audit**: Analyze missing values, anomalies, and duplicates.
3. **Documented Cleaning Decisions**: Explicitly justify every cleaning step using `# Decision:` and `# Reason:`.
4. **Outlier Investigation**: Eliminate impossible sensor failures while defending valid physical extremes.
5. **Before-and-After Snapshot**: Present a clear comparison table before and after cleaning.
6. **Clean Deliverable**: Export verified clean dataset (`cleaned_EV_Energy_Consumption.csv`).


In [1]:
# 1. Environment Setup & Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# # Visual settings
# plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
# plt.rcParams['figure.figsize'] = (10, 5)

# print("Environment configured and libraries imported successfully.")


In [3]:
# 2. Loading the Raw Dataset
raw_data_path = "../Data/EV_Energy_Consumption.csv"
df = pd.read_csv(raw_data_path)

print(f"Raw Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

Raw Dataset Shape: 5,022 rows, 19 columns


In [4]:
df.head()

,Vehicle_ID,Timestamp,Speed_kmh,Acceleration_ms2,Battery_State_%,Battery_Voltage_V,Battery_Temperature_C,Driving_Mode,Road_Type,Traffic_Condition,Slope_%,Weather_Condition,Temperature_C,Humidity_%,Wind_Speed_ms,Tire_Pressure_psi,Vehicle_Weight_kg,Distance_Travelled_km,Energy_Consumption_kWh
0,1102,2024-01-01 00:00:00,111.507366,-2.773816,30.415148,378.091525,25.314786,2,1,1,6.879446,4,0.741770,42.172533,7.829253,31.112020,1822.967368,20.757508,12.054317
1,1435,2024-01-01 00:01:00,48.612323,-0.796982,97.385534,392.718377,18.240755,1,2,1,-3.007212,4,-3.495516,57.018427,4.495572,31.504366,2091.831914,0.642918,4.488701
2,1860,2024-01-01 00:02:00,108.733320,0.253800,84.912600,398.993495,44.449145,1,1,3,0.029585,1,9.248275,69.028911,5.144489,33.838015,1816.702497,40.842824,11.701377
3,1270,2024-01-01 00:03:00,38.579484,-2.111395,28.777904,358.128273,28.980155,1,2,2,8.271943,3,2.868409,86.638349,4.518283,33.256014,1283.102642,5.305229,7.389266
4,1106,2024-01-01 00:04:00,57.172438,1.477883,29.740160,310.888162,33.184551,2,1,1,2.776814,2,16.750244,27.189185,4.263406,33.579678,2160.350788,5.825926,6.761205


In [ ]:
# 3. Data Overview & Summary Statistics (Before Cleaning)
print("--- DataFrame Schema Info ---")
df.info()

--- DataFrame Schema Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5022 entries, 0 to 5021
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Vehicle_ID              5022 non-null   int64  
 1   Timestamp               5021 non-null   object 
 2   Speed_kmh               5021 non-null   float64
 3   Acceleration_ms2        5022 non-null   float64
 4   Battery_State_%         5022 non-null   float64
 5   Battery_Voltage_V       5022 non-null   float64
 6   Battery_Temperature_C   5022 non-null   float64
 7   Driving_Mode            5022 non-null   int64  
 8   Road_Type               5022 non-null   int64  
 9   Traffic_Condition       5022 non-null   int64  
 10  Slope_%                 5021 non-null   float64
 11  Weather_Condition       5022 non-null   int64  
 12  Temperature_C           5021 non-null   float64
 13  Humidity_%              5021 non-null   float64
 14  Wind_Speed

In [5]:

print("Summary Statistics (Raw Data)")
display(df.describe().T.round(2))

Summary Statistics (Raw Data)


,count,mean,std,min,25%,50%,75%,max
Vehicle_ID,5022.0,1542.92,631.73,1000.00,1255.00,1512.00,1763.00,10012.0
Speed_kmh,5021.0,59.33,34.62,-20.00,29.59,58.62,89.00,250.0
Acceleration_ms2,5022.0,-0.01,1.74,-3.00,-1.49,-0.03,1.48,14.8
Battery_State_%,5022.0,60.21,23.16,-15.00,40.12,60.49,80.39,120.0
Battery_Voltage_V,5022.0,351.30,30.65,-50.00,326.76,351.65,376.19,920.0
Battery_Temperature_C,5022.0,27.52,10.12,-55.00,18.92,27.61,35.91,105.0
Driving_Mode,5022.0,2.03,1.59,1.00,1.00,2.00,3.00,99.0
Road_Type,5022.0,2.00,0.81,1.00,1.00,2.00,3.00,3.0
Traffic_Condition,5022.0,2.01,0.81,0.00,1.00,2.00,3.00,3.0
Slope_%,5021.0,2.51,4.40,-4.99,-1.23,2.45,6.35,48.0


In [16]:
# 4. Data Quality Audit: Missing Values & Duplicates Analysis
# 4.1 Missing values detection
missing_series = df.isnull().sum()
missing_table = pd.DataFrame({
    'Missing_Count': missing_series[missing_series > 0],
    'Missing_Percentage_%': (missing_series[missing_series > 0] / len(df) * 100).round(3)
})

print(f"Total Missing Values in Dataset: {df.isnull().sum().sum()}")



Total Missing Values in Dataset: 9


In [17]:
display(missing_table)

# 4.2 Exact duplicate rows check
duplicate_rows = df.duplicated().sum()
print(f"Total Exact Duplicate Rows: {duplicate_rows}")

,Missing_Count,Missing_Percentage_%
Timestamp,1,0.02
Speed_kmh,1,0.02
Slope_%,1,0.02
Temperature_C,1,0.02
Humidity_%,1,0.02
Wind_Speed_ms,1,0.02
Tire_Pressure_psi,1,0.02
Energy_Consumption_kWh,2,0.04


Total Exact Duplicate Rows: 0


In [5]:
# 5. Documented Data Cleaning Decisions

# Decision 1: Drop rows where the target variable 'Energy_Consumption_kWh' is missing.
# Reason: Target values cannot be synthetically imputed without introducing bias into downstream regression models.
df = df.dropna(subset=['Energy_Consumption_kWh'])

# Decision 2: Remove rows with physically impossible negative values in strictly positive domains.
# Reason: Negative speed (-20 km/h), negative distance (-5 km), negative voltage (-50 V), negative SoC (-15%), and negative humidity (-20%) are severe sensor errors violating physical laws.
df = df[
    (df['Speed_kmh'] >= 0) &
    (df['Distance_Travelled_km'] >= 0) &
    (df['Battery_Voltage_V'] > 0) &
    (df['Battery_State_%'] >= 0) & (df['Battery_State_%'] <= 100) &
    (df['Humidity_%'] >= 0) & (df['Humidity_%'] <= 100) &
    (df['Wind_Speed_ms'] >= 0)
]

# Decision 3: Remove records with invalid categorical codes.
# Reason: Driving_Mode has valid domain codes {1: Eco, 2: Normal, 3: Sport}, Traffic_Condition has {1, 2, 3}, and Weather_Condition has {1, 2, 3, 4}. Codes such as 99, 0, or -1 are undefined values.
df = df[
    df['Driving_Mode'].isin([1, 2, 3]) &
    df['Traffic_Condition'].isin([1, 2, 3]) &
    df['Weather_Condition'].isin([1, 2, 3, 4])
]

# Decision 4: Filter out impossible extreme outliers while defending and retaining physically valid extremes.
# Reason: Extreme sensor failures (Speed=250 km/h, Vehicle_Weight=75 kg or 5200 kg, Tire_Pressure=6 or 100 psi, Battery_Temp=105°C or -55°C, Distance=1250 km, Acc=14.8 m/s²) are hardware anomalies. High speeds within highway limits (up to 120 km/h) and normal curb weights (1200-2500 kg) are defended and retained.
df = df[
    (df['Speed_kmh'] <= 160) &
    (df['Acceleration_ms2'] >= -3.5) & (df['Acceleration_ms2'] <= 3.5) &
    (df['Battery_Voltage_V'] <= 500) &
    (df['Battery_Temperature_C'] >= -20) & (df['Battery_Temperature_C'] <= 60) &
    (df['Distance_Travelled_km'] <= 60) &
    (df['Wind_Speed_ms'] <= 40) &
    (df['Tire_Pressure_psi'] >= 20) & (df['Tire_Pressure_psi'] <= 45) &
    (df['Vehicle_Weight_kg'] >= 1000) & (df['Vehicle_Weight_kg'] <= 3000) &
    (df['Slope_%'] >= -15) & (df['Slope_%'] <= 15)
]

# Decision 5: Filter out physical and logical contradictions.
# Reason: A stationary vehicle (0 km/h) cannot log 160 km of travel distance, nor can a vehicle cruising at 110 km/h consume 0.0 kWh of energy.
df = df[~((df['Speed_kmh'] == 0) & (df['Distance_Travelled_km'] > 1))]
df = df[~((df['Speed_kmh'] > 50) & (df['Energy_Consumption_kWh'] <= 0.5))]

# Decision 6: Convert 'Timestamp' from string object to pandas datetime64.
# Reason: Enables chronological sorting, time-based queries, and extraction of temporal features (hour, day of week).
df = df[df['Timestamp'].notnull()]
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

print(f"Data Cleaning Completed. Remaining Rows: {len(df):,}")


Data Cleaning Completed. Remaining Rows: 5,000


In [6]:
# 6. Before-and-After Snapshot Comparison Table
snapshot_table = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Missing Values (NaN)',
        'Duplicate Rows',
        'Speed Min / Max (km/h)',
        'Battery SoC Range (%)',
        'Battery Voltage Range (V)',
        'Vehicle Weight Range (kg)',
        'Timestamp Data Type',
        'Dataset Trustworthiness'
    ],
    'Before (Raw Data)': [
        f"{len(df_raw):,}",
        f"{df_raw.shape[1]}",
        f"{df_raw.isnull().sum().sum()}",
        f"{df_raw.duplicated().sum()}",
        f"[{df_raw['Speed_kmh'].min()}, {df_raw['Speed_kmh'].max()}]",
        f"[{df_raw['Battery_State_%'].min()}, {df_raw['Battery_State_%'].max()}]",
        f"[{df_raw['Battery_Voltage_V'].min()}, {df_raw['Battery_Voltage_V'].max()}]",
        f"[{df_raw['Vehicle_Weight_kg'].min()}, {df_raw['Vehicle_Weight_kg'].max()}]",
        str(df_raw['Timestamp'].dtype),
        "Contains corrupted / anomalous records"
    ],
    'After (Cleaned Data)': [
        f"{len(df):,}",
        f"{df.shape[1]}",
        f"{df.isnull().sum().sum()}",
        f"{df.duplicated().sum()}",
        f"[{df['Speed_kmh'].min():.1f}, {df['Speed_kmh'].max():.1f}]",
        f"[{df['Battery_State_%'].min():.1f}, {df['Battery_State_%'].max():.1f}]",
        f"[{df['Battery_Voltage_V'].min():.1f}, {df['Battery_Voltage_V'].max():.1f}]",
        f"[{df['Vehicle_Weight_kg'].min():.1f}, {df['Vehicle_Weight_kg'].max():.1f}]",
        str(df['Timestamp'].dtype),
        "100% Validated & Clean"
    ]
})

display(snapshot_table)


,Metric,Before (Raw Data),After (Cleaned Data)
0,Total Rows,"5,022","5,000"
1,Total Columns,19,19
2,Missing Values (NaN),9,0
3,Duplicate Rows,0,0
4,Speed Min / Max (km/h),"[-20.0, 250.0]","[0.0, 119.9]"
5,Battery SoC Range (%),"[-15.0, 120.0]","[20.0, 100.0]"
6,Battery Voltage Range (V),"[-50.0, 920.0]","[300.0, 400.0]"
7,Vehicle Weight Range (kg),"[75.0, 5200.0]","[1201.0, 2499.5]"
8,Timestamp Data Type,object,datetime64[ns]
9,Dataset Trustworthiness,Contains corrupted / anomalous records,100% Validated & Clean


In [7]:
# 7. Summary Statistics of Cleaned Dataset
print("--- Summary Statistics (Cleaned Data) ---")
display(df.describe().T.round(2))


--- Summary Statistics (Cleaned Data) ---


,count,mean,min,25%,50%,75%,max,std
Vehicle_ID,5000.0,1505.7012,1000.0,1253.0,1510.0,1761.0,1999.0,290.836902
Timestamp,5000,2024-01-02 17:39:30,2024-01-01 00:00:00,2024-01-01 20:49:45,2024-01-02 17:39:30,2024-01-03 14:29:15,2024-01-04 11:19:00,NaN
Speed_kmh,5000.0,59.312952,0.001396,29.5154,58.646794,89.030001,119.946924,34.517605
Acceleration_ms2,5000.0,-0.019885,-2.999683,-1.49024,-0.043329,1.473887,2.998039,1.724287
Battery_State_%,5000.0,60.214465,20.020196,40.088318,60.490283,80.409928,99.993986,23.128262
Battery_Voltage_V,5000.0,351.118306,300.004812,326.613439,351.510915,375.96814,399.990098,28.968372
Battery_Temperature_C,5000.0,27.51013,10.003858,18.884742,27.58171,35.920162,44.985943,10.015385
Driving_Mode,5000.0,2.011,1.0,1.0,2.0,3.0,3.0,0.810068
Road_Type,5000.0,1.9992,1.0,1.0,2.0,3.0,3.0,0.808536
Traffic_Condition,5000.0,2.0108,1.0,1.0,2.0,3.0,3.0,0.812167


In [8]:
# 8. Exporting the Final Cleaned Dataset
cleaned_data_path = "../Data/cleaned_EV_Energy_Consumption.csv"
df.to_csv(cleaned_data_path, index=False)

# Verification
assert os.path.exists(cleaned_data_path), "Exported file not found!"
df_verify = pd.read_csv(cleaned_data_path)

print(f"Cleaned dataset successfully saved to: {cleaned_data_path}")
print(f"Verified Dimensions: {df_verify.shape[0]:,} rows, {df_verify.shape[1]} columns")
print(f"Verified Missing Values: {df_verify.isnull().sum().sum()}")


Cleaned dataset successfully saved to: ../Data/cleaned_EV_Energy_Consumption.csv
Verified Dimensions: 5,000 rows, 19 columns
Verified Missing Values: 0
